# Fitur otomatis (mini OpenFE) + fitur tetangga terdekat (kNN OOF)
**Kapan dipakai:** data teks + tabular untuk GBDT (LightGBM) dan waktu untuk merancang fitur
manual habis. Dua mesin fitur:

1. **Fitur tetangga (kNN OOF):** di ruang SVD teks, porsi tiap kelas di antara k tetangga
   terdekat dan jarak ke tetangga terdekat tiap kelas. Dihitung out-of-fold (trik juara Kaggle
   Otto Group 2015). Membawa "teks mirip punya label apa" ke model tabular.
2. **Mini OpenFE** (Zhang dkk., ICML 2023): buat ratusan kandidat (a+b, a−b, a×b, a/b,
   rata-rata per kelompok, selisih dari rata-rata kelompok, frekuensi), lalu nilai tiap kandidat
   dengan **FeatureBoost**: model kecil 1 fitur dilatih di atas tebakan model dasar
   (`init_score`), jadi yang diukur hanya informasi TAMBAHAN. Kandidat disaring dengan
   **successive halving** (25% → 50% → 100% baris, separuh terbaik lolos tiap tahap).

**Jujur:** pemilihan kandidat dan CV akhir memakai train yang sama → skor sedikit optimis.
Pakai fitur baru hanya kalau naiknya ≥ `MIN_GAIN`.

In [ ]:
import os
import time

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/fitur_otomatis_tetangga"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
ID_COL = "id"                                # GANTI saat lomba
TARGET = "recommended_ind"                   # GANTI saat lomba (`rating` bocor, tidak dipakai)
NUM_COLS = ["age", "positive_feedback_count"]                               # GANTI saat lomba
CAT_COLS = ["clothing_id", "division_name", "department_name", "class_name"]  # GANTI saat lomba
TEXT_COLS = ["title", "review_text"]                                        # GANTI saat lomba
N_SVD = 64                   # ruang tetangga, 16 komponen pertama juga jadi fitur dasar
KNN_SIZES = [5, 25]
HALVING_STEPS = [0.25, 0.5, 1.0]
N_KEEP = 20
MIN_GAIN = 0.001
N_FOLDS = 5
SEED = 42


def metric(truth, proba):
    return roc_auc_score(truth, proba[:, 1])       # GANTI: metrik lomba (proba = n × K)


def make_model():
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                              random_state=SEED, verbose=-1)


print("uji metrik:", metric(np.array([0, 1]), np.array([[0.8, 0.2], [0.3, 0.7]])))

## Baca data + fitur dasar

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
both = pd.concat([train, test], ignore_index=True)
n_train = len(train)
n_test = len(test)
CLASSES, labels_all = np.unique(train[TARGET], return_inverse=True)
base = pd.DataFrame(index=both.index)
text_stat_cols = []
for col in NUM_COLS:
    base[col] = both[col]
for col in CAT_COLS:
    base[col] = both[col].astype("category").cat.codes
for col in TEXT_COLS:
    text = both[col].fillna("").astype(str)
    base[f"{col}_huruf"] = text.str.len()
    base[f"{col}_kata"] = text.str.split().str.len()
    base[f"{col}_seru"] = text.str.count("!")
    text_stat_cols += [f"{col}_huruf", f"{col}_kata", f"{col}_seru"]
full_text = both[TEXT_COLS].fillna("").agg(" . ".join, axis=1)
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000,
                        sublinear_tf=True).fit_transform(full_text)
vectors = normalize(TruncatedSVD(N_SVD, random_state=SEED).fit_transform(tfidf))
for number in range(16):
    base[f"svd_{number}"] = vectors[:, number]
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train, labels_all))
print("fitur dasar:", base.shape, "| kelas:", list(CLASSES))

## 1. Fitur tetangga (kNN OOF)
Train: tetangga dicari di lipatan lain (OOF). Test: tetangga dari seluruh train.

In [ ]:
def neighbor_features(fit_rows, query_rows):
    out = pd.DataFrame(index=np.arange(len(query_rows)))
    finder = NearestNeighbors(n_neighbors=max(KNN_SIZES)).fit(vectors[fit_rows])
    _, found = finder.kneighbors(vectors[query_rows])
    neighbor_labels = labels_all[fit_rows][found]
    for size in KNN_SIZES:
        for number, name in enumerate(CLASSES):
            out[f"knn{size}_{name}"] = (neighbor_labels[:, :size] == number).mean(axis=1)
    for number, name in enumerate(CLASSES):
        members = fit_rows[labels_all[fit_rows] == number]
        nearest = NearestNeighbors(n_neighbors=3).fit(vectors[members])
        distance, _ = nearest.kneighbors(vectors[query_rows])
        out[f"jarak_{name}"] = distance.mean(axis=1)
    return out


start = time.time()
parts = []
for idx_fit, idx_val in folds:
    part = neighbor_features(idx_fit, idx_val)
    part.index = idx_val
    parts.append(part)
knn_train = pd.concat(parts).sort_index()
knn_test = neighbor_features(np.arange(n_train), np.arange(n_train, n_train + n_test))
print("fitur tetangga:", list(knn_train.columns), "| detik", round(time.time() - start))

## 2a. Kandidat fitur otomatis

In [ ]:
def arithmetic_candidates(frame, columns):
    found = {}
    for first_number, first in enumerate(columns):
        for second in columns[first_number + 1:]:
            left = frame[first].to_numpy(float)
            right = frame[second].to_numpy(float)
            found[f"{first}+{second}"] = left + right
            found[f"{first}-{second}"] = left - right
            found[f"{first}*{second}"] = left * right
            found[f"{first}/{second}"] = left / (np.abs(right) + 1e-6)
    return found


def group_candidates(frame, cats, numbers):
    found = {}
    for cat in cats:
        found[f"freq({cat})"] = frame[cat].map(frame[cat].value_counts()).to_numpy(float)
        for number_col in numbers:
            grouped = frame.groupby(cat)[number_col]
            group_mean = grouped.transform("mean").to_numpy(float)
            found[f"mean({number_col}|{cat})"] = group_mean
            found[f"{number_col}-mean|{cat}"] = frame[number_col].to_numpy(float) - group_mean
    return found


numeric_base = NUM_COLS + text_stat_cols
candidates = arithmetic_candidates(base, numeric_base)
candidates.update(group_candidates(base, CAT_COLS, numeric_base))
print("kolom angka dasar:", numeric_base, "| kandidat:", len(candidates))

## 2b. Tebakan model dasar (skor mentah OOF) untuk FeatureBoost

In [ ]:
def raw_to_proba(raw):
    if raw.ndim == 1:
        return np.c_[1 - expit(raw), expit(raw)]
    shifted = np.exp(raw - raw.max(axis=1, keepdims=True))
    return shifted / shifted.sum(axis=1, keepdims=True)


base_raw = np.zeros(n_train)
if len(CLASSES) > 2:
    base_raw = np.zeros((n_train, len(CLASSES)))
base_values = base.iloc[:n_train].to_numpy(float)
for idx_fit, idx_val in folds:
    model = make_model().fit(base_values[idx_fit], labels_all[idx_fit])
    base_raw[idx_val] = model.predict(base_values[idx_val], raw_score=True)
print("skor model dasar (OOF):", round(metric(labels_all, raw_to_proba(base_raw)), 4))

## 2c. FeatureBoost + successive halving
Untung kandidat = logloss dasar − logloss setelah model kecil 1 fitur ditambahkan di atas
tebakan dasar. Positif = kandidat membawa informasi baru.

In [ ]:
def boost_gain(values, split):
    fit_rows, val_rows = split
    booster = lgb.LGBMClassifier(n_estimators=30, learning_rate=0.1, num_leaves=7,
                                 random_state=SEED, verbose=-1)
    booster.fit(values[fit_rows].reshape(-1, 1), labels_all[fit_rows],
                init_score=base_raw[fit_rows])
    raw = booster.predict(values[val_rows].reshape(-1, 1), raw_score=True) + base_raw[val_rows]
    every = np.arange(len(CLASSES))
    before = log_loss(labels_all[val_rows], raw_to_proba(base_raw[val_rows]), labels=every)
    return before - log_loss(labels_all[val_rows], raw_to_proba(raw), labels=every)


rng = np.random.default_rng(SEED)
shuffled = rng.permutation(n_train)
survivors = list(candidates)
gains = {}
start = time.time()
for fraction in HALVING_STEPS:
    rows = shuffled[: int(fraction * n_train)]
    split = (rows[: len(rows) // 2], rows[len(rows) // 2:])
    gains = {}
    for name in survivors:
        gains[name] = boost_gain(candidates[name][:n_train], split)
    survivors = sorted(survivors, key=gains.get, reverse=True)[: max(N_KEEP, len(survivors) // 2)]
    print(f"tahap {fraction:.0%}: {len(gains)} kandidat dinilai, {len(survivors)} lolos")
print("detik:", round(time.time() - start))

## 2d. Kandidat terpilih (bahan slide)
Kandidat yang hanya "kode ulang" kandidat lain (mis. semua `mean(x|kelompok)` dengan kelompok
sama membagi baris persis sama) dibuang: tanda tangan = hasil `pd.factorize`.

In [ ]:
chosen_new = []
seen = set()
for name in survivors:
    signature = hash(pd.factorize(candidates[name][:n_train])[0].tobytes())
    if gains[name] > 0 and signature not in seen and len(chosen_new) < N_KEEP:
        chosen_new.append(name)
        seen.add(signature)
gain_table = pd.DataFrame({"fitur": chosen_new})
gain_table["untung logloss"] = gain_table["fitur"].map(gains).round(5)
gain_table.to_csv(f"{OUTPUT_FOLDER}/fitur_otomatis_terpilih.csv", index=False)
print(gain_table.to_string(index=False))
auto_all = pd.DataFrame(index=both.index)
for name in chosen_new:
    auto_all[name] = candidates[name]

## 3. Banding akhir: dasar vs + tetangga vs + otomatis vs semua (CV 5 lipatan)

In [ ]:
def cv_score(frame):
    values = frame.to_numpy(float)
    oof = np.zeros((n_train, len(CLASSES)))
    for idx_fit, idx_val in folds:
        model = make_model().fit(values[idx_fit], labels_all[idx_fit])
        oof[idx_val] = model.predict_proba(values[idx_val])
    return metric(labels_all, oof)


train_base = base.iloc[:n_train].reset_index(drop=True)
train_auto = auto_all.iloc[:n_train].reset_index(drop=True)
SETS = {"dasar": train_base, "+ tetangga": pd.concat([train_base, knn_train], axis=1),
        "+ otomatis": pd.concat([train_base, train_auto], axis=1),
        "+ keduanya": pd.concat([train_base, knn_train, train_auto], axis=1)}
rows = []
for name, frame in SETS.items():
    rows.append({"fitur": name, "jumlah kolom": frame.shape[1], "CV": round(cv_score(frame), 4)})
    print(rows[-1])
table = pd.DataFrame(rows)
table.to_csv(f"{OUTPUT_FOLDER}/banding_fitur.csv", index=False)

## Simpan fitur baru (train + test, kunci `ID_COL`) + gambar
File ini bisa digabung ke notebook LightGBM lain (`stacking_teks_tabular`) lewat `pd.concat`
berdasarkan urutan baris yang sama.

In [ ]:
new_train = pd.concat([train[[ID_COL]], knn_train, train_auto], axis=1)
test_auto = auto_all.iloc[n_train:].reset_index(drop=True)
new_test = pd.concat([test[[ID_COL]], knn_test, test_auto], axis=1)
new_train.to_csv(f"{OUTPUT_FOLDER}/fitur_baru_train.csv", index=False)
new_test.to_csv(f"{OUTPUT_FOLDER}/fitur_baru_test.csv", index=False)
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].barh(table["fitur"], table["CV"], color="#0072B2")
axes[0].set_xlim(table["CV"].min() - 0.005, table["CV"].max() + 0.003)
axes[0].set_xlabel("CV")
axes[0].invert_yaxis()
axes[1].barh(gain_table["fitur"][:12], gain_table["untung logloss"][:12], color="#E69F00")
axes[1].set_xlabel("untung logloss (FeatureBoost)")
axes[1].invert_yaxis()
axes[1].tick_params(axis="y", labelsize=7)
figure.tight_layout()
figure.savefig(f"{OUTPUT_FOLDER}/fitur_baru.png", dpi=200, bbox_inches="tight")
plt.show()
print("ditulis:", new_train.shape, new_test.shape)